# Multi-task ablations

Issue: #28

## Question

Does the multi-task design (MMoE, three tasks trained together) actually help, compared with simpler alternatives, especially for the rare **order** task?

## Data used

The same feature tables, split, features, seed, batch size (4096) and epoch budget (10, best epoch by validation loss) as #27; code in `notebooks/dcn.py`.

Four models with the same DCN-V2 cross layers:
- **MMoE** (from #27, loaded): 4 experts, one gate and tower per task, trained on all three tasks.
- **Shared-bottom**: one expert shared by all three tasks, one tower per task, trained on all three tasks. No gates, so the tasks cannot choose different features.
- **Single-task** (three models): one model per task (click, cart, order), each trained on its own label only. Their probabilities are blended with the same weights afterwards.

The blended score uses OTTO's weights (0.10, 0.30, 0.60) for all of them. Loss weights for the two multi-task models are click 1, cart 2, order 4.

In [ ]:
import json
import resource
import sys
from pathlib import Path

import numpy as np
import polars as pl
import torch

sys.path.insert(0, ".")
sys.path.insert(0, "notebooks")
from bootstrap import paired_bootstrap_diff
from dcn import DCNMTL, OTTO_WEIGHTS, blend, predict, task_auc, train_model
from evalset import TOP, evaluate, prepare_eval, summarize
from features import FEATURE_COLUMNS

torch.set_num_threads(4)
DATA = Path("data") if Path("data").exists() else Path("../data")
FEAT, RESULTS = DATA / "features", DATA / "results"
POOL, SEED, EPOCHS = 200, 0, 10
NUM = [c for c in FEATURE_COLUMNS if c not in ("category", "category_conf")]
TASKS = ["y_click", "y_cart", "y_order"]
NAMES = ["click", "cart", "order"]
cfg = json.loads((DATA / "models" / "dcn_mmoe" / "config.json").read_text())

In [ ]:
tr_all = pl.read_parquet(FEAT / "train_asof11.parquet")
sessions = tr_all["session"].unique().sort().to_numpy()
rng = np.random.default_rng(SEED)
valid_sessions = set(rng.choice(sessions, size=len(sessions) // 10, replace=False).tolist())
is_val = tr_all["session"].is_in(list(valid_sessions)).to_numpy()
tr_part = tr_all.filter(~pl.Series(is_val))
mean = tr_part.select(NUM).mean().to_numpy().ravel().astype(np.float32)
std = tr_part.select(NUM).std().to_numpy().ravel().astype(np.float32)
std[std < 1e-6] = 1.0
# same split and scaling as #27
assert np.allclose(mean, np.array(cfg["mean"], dtype=np.float32)) and np.allclose(std, np.array(cfg["std"], dtype=np.float32))


def tensors(df):
    x = torch.from_numpy((df.select(NUM).to_numpy().astype(np.float32) - mean) / std)
    c = torch.from_numpy(df["category"].to_numpy().astype(np.int64))
    y = torch.from_numpy(df.select(TASKS).to_numpy().astype(np.float32))
    return x, c, y


x_tr, c_tr, y_tr = tensors(tr_part)
x_va, c_va, y_va = tensors(tr_all.filter(pl.Series(is_val)))
del tr_all, tr_part
n_cats = cfg["n_cats"]
print(f"train rows {len(x_tr):,}, validation rows {len(x_va):,}; positives per task (train): {y_tr.sum(0).int().tolist()}")

## Train the four models

In [ ]:
models = {}
info = {}

mmoe = DCNMTL(n_num=len(NUM), n_cats=n_cats, n_tasks=3, mode="mmoe")
mmoe.load_state_dict(torch.load(DATA / "models" / "dcn_mmoe" / "model.pt"))
models["MMoE (#27)"] = [mmoe]
info["MMoE (#27)"] = {"params": sum(p.numel() for p in mmoe.parameters()), "seconds": cfg["train_seconds"], "best_epoch": cfg["best_epoch"]}

shared = DCNMTL(n_num=len(NUM), n_cats=n_cats, n_tasks=3, mode="shared")
hist, best, secs = train_model(shared, x_tr, c_tr, y_tr, x_va, c_va, y_va, task_weights=(1.0, 2.0, 4.0), epochs=EPOCHS, label="shared-bottom")
models["Shared-bottom"] = [shared]
info["Shared-bottom"] = {"params": sum(p.numel() for p in shared.parameters()), "seconds": round(secs), "best_epoch": best}

singles = []
tot_secs, tot_params = 0, 0
for t, name in enumerate(NAMES):
    m = DCNMTL(n_num=len(NUM), n_cats=n_cats, n_tasks=1, mode="shared")
    hist, best, secs = train_model(m, x_tr, c_tr, y_tr[:, t : t + 1], x_va, c_va, y_va[:, t : t + 1], task_weights=(1.0,), epochs=EPOCHS, label=f"single {name}")
    singles.append(m)
    tot_secs += secs
    tot_params += sum(p.numel() for p in m.parameters())
models["Single-task x3"] = singles
info["Single-task x3"] = {"params": tot_params, "seconds": round(tot_secs), "best_epoch": "per model"}
print(info)

## Predictions with the same interface for all four

In [ ]:
def probs(model_list, x, c):
    if len(model_list) == 1:
        return predict(model_list[0], x, c)
    return np.concatenate([predict(m, x, c) for m in model_list], axis=1)   # columns: click, cart, order


ev = pl.read_parquet(FEAT / "eval_w0.parquet")
prepared, prep_stats = prepare_eval(pl.read_parquet(DATA / "eval" / "ranking_queries.parquet"))
n_q = len(prepared)
assert ev.height == n_q * POOL and ev["session"].to_numpy()[::POOL].tolist() == [q["session"] for q in prepared]
x_ev, c_ev, y_ev = tensors(ev)
aids = ev["aid"].to_numpy().reshape(n_q, POOL)
row_of = {q["session"]: i for i, q in enumerate(prepared)}

p_val, p_eval, results = {}, {}, {}
for name, ml in models.items():
    p_val[name] = probs(ml, x_va, c_va)
    p_eval[name] = probs(ml, x_ev, c_ev)
    assert p_val[name].shape[1] == 3 and p_eval[name].shape[1] == 3


def ranker(p):
    scores = blend(p, OTTO_WEIGHTS).reshape(n_q, POOL)
    order = np.argsort(-scores, axis=1, kind="stable")[:, :TOP]
    ranked = np.take_along_axis(aids, order, axis=1)
    return lambda q: [int(a) for a in ranked[row_of[q["session"]]]]


for name in models:
    results[name] = evaluate(prepared, ranker(p_eval[name]))
print("evaluated", len(prepared), "queries with", len(models), "models")

## Comparison: per-task AUC and blended NDCG@10

In [ ]:
ref = "MMoE (#27)"
rows = []
for name in models:
    auc_v = task_auc(p_val[name], y_va.numpy())
    auc_e = task_auc(p_eval[name], y_ev.numpy())
    s = summarize(results[name])
    j = results[name].join(results[ref].select("session", pl.col("ndcg10").alias("ref")), on="session")
    diff = paired_bootstrap_diff(j["ndcg10"].to_numpy(), j["ref"].to_numpy()) if name != ref else (0.0, 0.0, 0.0)
    rows.append({
        "model": name, "params": info[name]["params"], "train_s": info[name]["seconds"],
        "auc_click_val": auc_v[0], "auc_cart_val": auc_v[1], "auc_order_val": auc_v[2],
        "auc_click_eval": auc_e[0], "auc_cart_eval": auc_e[1], "auc_order_eval": auc_e[2],
        "ndcg10": s["ndcg10"][0], "ndcg10_low": s["ndcg10"][1], "ndcg10_high": s["ndcg10"][2],
        "diff_vs_mmoe": diff[0], "diff_low": diff[1], "diff_high": diff[2],
    })
table = pl.DataFrame(rows)
with pl.Config(tbl_cols=20, tbl_width_chars=250, fmt_float="mixed"):
    print(table.select("model", "params", "train_s", "auc_click_val", "auc_cart_val", "auc_order_val"))
    print(table.select("model", "auc_click_eval", "auc_cart_eval", "auc_order_eval"))
    print(table.select("model", "ndcg10", "ndcg10_low", "ndcg10_high", "diff_vs_mmoe", "diff_low", "diff_high"))

In [ ]:
# pairwise: multi-task shared-bottom vs single-task
j = results["Shared-bottom"].join(results["Single-task x3"].select("session", pl.col("ndcg10").alias("o")), on="session")
m_, lo, hi = paired_bootstrap_diff(j["ndcg10"].to_numpy(), j["o"].to_numpy())
print(f"NDCG@10 shared-bottom minus single-task: {m_:+.4f}  [{lo:+.4f}, {hi:+.4f}]")
order_auc = table.select("model", "auc_order_val", "auc_order_eval")
print(order_auc)
print("peak memory GB:", round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e9, 2))
assert len({tuple(y_tr.shape), tuple(y_va.shape)}) == 2   # all models saw the same tensors

## Result

Same data, split, seed, batch size and epoch budget for every model. Training rows have 31,374 click, 2,275 cart and only **523 order** positives.

**Per-task AUC** (validation / evaluation):

| Model | Parameters | click | cart | order |
|---|---|---|---|---|
| MMoE (#27) | 32,961 | 0.851 / 0.865 | **0.904 / 0.900** | **0.928 / 0.892** |
| Shared-bottom | 14,973 | 0.847 / 0.864 | 0.895 / 0.888 | 0.905 / 0.874 |
| Single-task x3 | 32,241 | 0.851 / 0.868 | 0.882 / 0.878 | 0.868 / 0.801 |

**Blended NDCG@10** (OTTO weights, 56,544 evaluation queries, mean [95% CI], paired difference to MMoE):

| Model | NDCG@10 | difference to MMoE |
|---|---|---|
| MMoE (#27) | 0.2069 [0.2043, 0.2096] | reference |
| Shared-bottom | 0.1994 [0.1969, 0.2019] | -0.0075 [-0.0086, -0.0065] |
| Single-task x3 | **0.2115** [0.2089, 0.2141] | **+0.0046 [+0.0037, +0.0055]** |

- **Multi-task learning helps the rare tasks.** For the order task (523 positives), MMoE has clearly higher AUC than three separate models: 0.928 vs 0.868 on validation and 0.892 vs 0.801 on evaluation, with the shared-bottom in between. The cart task shows the same order (0.904 > 0.895 > 0.882). For clicks, which have plenty of data, all three are about equal (0.851 to 0.868). This is the case that multi-task learning is meant for: a task with few positives borrows what the other tasks learn.
- **Multi-task did not help the final ranking.** The three single-task models give the best NDCG@10 (+0.0046 over MMoE, interval excludes zero), and the shared-bottom is the worst (-0.0075). NDCG@10 is decided mainly by the click and cart probabilities near the top of the list, where the single-task click model is as good or better, and the blended score puts 0.60 of its weight on the order probability, which is estimated from very few examples and is noisy whatever the model.
- **Shared-bottom is worse than MMoE** on every task and on NDCG: with one shared expert, the tasks cannot pick different parts of the input, and the click task, which dominates the loss, pulls the shared layer. The gates of MMoE let each task weight the experts differently, and it costs about twice the parameters.
- **Caution.** The order AUC on validation rests on very few positives (the validation set has 4,000 queries, so only a few dozen orders); the wide gap between validation and evaluation values (e.g. single-task 0.868 vs 0.801) shows how noisy it is. Only the NDCG intervals, computed over 56,544 queries, are tight.
- **Answer to the question:** yes for the rare order task's AUC, no for the end metric. The single-task models are also the simplest, but they cost three trainings (200 s in total vs 241 s for MMoE here, so no saving) and give three separate models to serve.

## What I learned

To be written by the owner of the project.